# Multi-Query Retriever

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

# this library is used for multi-query retriever
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

In [4]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [7]:
# Initialize HuggingFace embeddings
embedding_model = HuggingFaceEmbeddings()

# Create FAISS vector store
vector_store = FAISS.from_documents(
    documents=all_docs,
    embedding=embedding_model
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 493.95it/s]


In [8]:
# Create simple similarity retriever for comparision between multi-query retriever
similarity_retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":5})

In [ ]:
# Initialize Hugging Face Endpoint and Chat Model
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

llm = HuggingFacePipeline.from_model_id(
    model_id="Qwen/Qwen2.5-1.5B-Instruct",
    task="text-generation",
    pipeline_kwargs={"max_new_tokens": 256, "temperature": 0.3, "do_sample": True},
)
chat_model = ChatHuggingFace(llm=llm)

Loading weights: 100%|██████████| 338/338 [00:01<00:00, 191.85it/s]


In [33]:
# Now Create multi-query retriever
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vector_store.as_retriever(search_kwargs={"k":5}),
    llm= chat_model                           # HuggingFace LLM is used to generate multiple query
)

In [34]:
# Query
query = "How to improve energy levels and maintain balance?"


# Retrieve results
# Here we comparing results of both retriever 
similarity_results = similarity_retriever.invoke(query)
multiquery_results = multiquery_retriever.invoke(query)

In [39]:
# print results

print("Similarity Retriever")
for i, doc in enumerate(similarity_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("\n")
print("*"*150)
print("\n")

print("Multi-Query Retriever")
for i, doc in enumerate(multiquery_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

Similarity Retriever

--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
Regular walking boosts heart health and can reduce symptoms of depression.

--- Result 3 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 4 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 5 ---
The solar energy system in modern homes helps balance electricity demand.


******************************************************************************************************************************************************


Multi-Query Retriever

--- Result 1 ---
Python balances readability with power, making it a popular system design language.

--- Result 2 ---
Deep sleep is crucial for cellular repair and emotional regulation.

--- Result 3 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 4 ---
Photosynthesis 